In [101]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 
import lightgbm as lgb
import pickle
from sklearn.preprocessing import LabelEncoder
import datetime

In [216]:
R = "11"
PLACE = "09"
DATE = "221123"
RaceID = "20" + DATE[:2] + "_" + DATE[2:4] + "_" + DATE[4:6] + "_" + PLACE + "_" + str(int(R)) + "R"
before_url = "https://www.boatrace.jp/owpc/pc/race/beforeinfo?rno=" + R + "&jcd=" + PLACE + "&hd=20" + DATE
before_info = pd.read_html(before_url)
tenji = np.array(before_info[1]['展示 タイム']['展示 タイム'])[::4]

In [217]:
tenji

array([6.73, 6.75, 6.92, 6.75, 6.81, 6.82])

In [218]:
CSV_PATH = "../csv/"
K_COLS = ['着', 'RaceID', '選手登番', '展示']

In [219]:
k_files = glob.glob("../csv/K" + DATE + ".csv")
b_files = glob.glob("../csv/B" + DATE + ".csv")
def concat_files(files):
    tmp = pd.DataFrame()
    for file in tqdm(files):
        df = pd.read_csv(file, index_col=0)
        tmp = pd.concat([tmp,df])
    return tmp
kdf = concat_files(k_files)
bdf = concat_files(b_files)

100%|██████████| 1/1 [00:00<00:00, 102.64it/s]


In [220]:
df = pd.merge(bdf,kdf.loc[:,K_COLS],on = ['RaceID','選手登番'])

In [221]:
df = bdf[bdf["RaceID"] == RaceID]
df["展示"] = tenji

/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/396100155.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["展示"] = tenji


In [222]:
df

,艇番,選手登番,選手名,年齢,支部,体重,級別,全国勝率,全国2連率,当地勝率,当地2連率,モーター2連率,ボート2連率,RaceID,場所,R,展示
564,1,3475,橋本久和,52,群馬,52,A2,5.49,35.87,7.00,50.00,34.48,3.57,2022_11_23_09_11R,9,11R,6.73
565,2,4740,齋藤達希,30,静岡,53,B1,3.88,16.05,4.08,22.95,28.26,33.33,2022_11_23_09_11R,9,11R,6.75
566,3,4491,田中堅,35,群馬,52,B1,4.32,19.82,4.78,24.44,24.00,45.24,2022_11_23_09_11R,9,11R,6.92
567,4,4824,松井洪弥,29,三重,51,A2,6.47,47.42,5.88,40.77,27.91,32.56,2022_11_23_09_11R,9,11R,6.75
568,5,4895,門間雄大,31,東京,54,B1,4.00,17.81,6.53,41.18,18.60,47.37,2022_11_23_09_11R,9,11R,6.81
569,6,3873,別府昌樹,46,広島,52,A2,5.96,41.25,0.00,0.00,35.14,34.78,2022_11_23_09_11R,9,11R,6.82


In [223]:
def LabelEncoding(df,col):
    #インスタンス
    LE = LabelEncoder()

    #Label エンコーディング
    LE.fit_transform(df[col].values)

    pickle.dump(LE, open('../model/' + col + '_LE.pickle', 'wb'))

    #データ変換
    le = LE.fit_transform(df[col].values)
    return le

def zero_padding(txt):
    l = re.findall(r"\d+", txt)
    l = [int(s) for s in l]
    date = datetime.date(*l[:3])
    raceid = str(date) + '-' + str(l[3]).zfill(2) + '-' + str(l[4]).zfill(2)
    return raceid

In [224]:
df_encoded = df
df_encoded['支部'] = LabelEncoding(df,'支部')
df_encoded['級別'] = LabelEncoding(df, '級別')
df_encoded['R'] = df['R'].replace('R', '', regex=True).astype('int')
df_encoded['RaceID'] = df_encoded['RaceID'].replace('_','/',regex=True).replace('R','',regex=True)

/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/1935421802.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_encoded['支部'] = LabelEncoding(df,'支部')
/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/1935421802.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_encoded['級別'] = LabelEncoding(df, '級別')
/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/1935421802.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFram

In [225]:
zero = []
for n,i in enumerate(tqdm(df_encoded['RaceID'])):
    zero.append(zero_padding(i))
df_encoded['RaceID'] = zero

100%|██████████| 6/6 [00:00<00:00, 7373.52it/s]
/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/536736072.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_encoded['RaceID'] = zero


In [226]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6 entries, 564 to 569
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   艇番       6 non-null      int64  
 1   選手登番     6 non-null      int64  
 2   選手名      6 non-null      object 
 3   年齢       6 non-null      int64  
 4   支部       6 non-null      int64  
 5   体重       6 non-null      int64  
 6   級別       6 non-null      int64  
 7   全国勝率     6 non-null      float64
 8   全国2連率    6 non-null      float64
 9   当地勝率     6 non-null      float64
 10  当地2連率    6 non-null      float64
 11  モーター2連率  6 non-null      float64
 12  ボート2連率   6 non-null      float64
 13  RaceID   6 non-null      object 
 14  場所       6 non-null      int64  
 15  R        6 non-null      int64  
 16  展示       6 non-null      float64
dtypes: float64(7), int64(8), object(2)
memory usage: 864.0+ bytes


In [227]:
df_test = df_encoded.drop(['選手名','RaceID'],axis=1)

In [228]:
lgb_clf = pickle
with open('../model/lgb_clf.pickle', 'rb') as f:
    lgb_clf = pickle.load(f)

In [229]:
df_pred = lgb_clf.predict(df_test ,num_iteration=lgb_clf.best_iteration)

In [230]:
df_encoded["pred"] = df_pred


/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/144378269.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_encoded["pred"] = df_pred


In [231]:
df_encoded

,艇番,選手登番,選手名,年齢,支部,体重,級別,全国勝率,全国2連率,当地勝率,当地2連率,モーター2連率,ボート2連率,RaceID,場所,R,展示,pred
564,1,3475,橋本久和,52,3,52,0,5.49,35.87,7.00,50.00,34.48,3.57,2022-11-23-09-11,9,11,6.73,2.039194
565,2,4740,齋藤達希,30,4,53,1,3.88,16.05,4.08,22.95,28.26,33.33,2022-11-23-09-11,9,11,6.75,-0.819822
566,3,4491,田中堅,35,3,52,1,4.32,19.82,4.78,24.44,24.00,45.24,2022-11-23-09-11,9,11,6.92,-0.966970
567,4,4824,松井洪弥,29,0,51,0,6.47,47.42,5.88,40.77,27.91,32.56,2022-11-23-09-11,9,11,6.75,0.020875
568,5,4895,門間雄大,31,2,54,1,4.00,17.81,6.53,41.18,18.60,47.37,2022-11-23-09-11,9,11,6.81,-1.611320
569,6,3873,別府昌樹,46,1,52,0,5.96,41.25,0.00,0.00,35.14,34.78,2022-11-23-09-11,9,11,6.82,-1.198704


In [232]:
pred_rank = pd.DataFrame()
for i in df_encoded["RaceID"].unique():
    prd = df_encoded[df_encoded["RaceID"] == i]["pred"].rank(ascending=False)
    pred_rank = pd.concat([pred_rank,prd])

In [233]:
df_encoded["pred_rank"] = pred_rank

/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_27719/3625637642.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_encoded["pred_rank"] = pred_rank


In [234]:
places = df_encoded['場所'].unique()
place_list = {
            '桐生':'01',
            '戸田':'02',
            '江戸川':'03',
            '平和島':'04',
            '多摩川':'05',
            '浜名湖':'06',
            '蒲郡':'07',
            '常滑':'08',
            '津':'09',
            '三国':'10',
            'びわこ':'11',
            '住之江':'12',
            '尼崎':'13',
            '鳴門':'14',
            '丸亀':'15',
            '児島':'16',
            '宮島':'17',
            '徳山':'18',
            '下関':'19',
            '若松':'20',
            '芦屋':'21',
            '福岡':'22',
            '唐津':'23',
            '大村':'24'}
def get_keys_from_value(d, val):
    return [k for k, v in d.items() if v == val]
for p in places:
    if p < 10:
        v = '0' + str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    else:
        v = str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    os.makedirs('../prediction/' + DATE + '/',exist_ok=True)
    with open("../prediction/"+DATE+"/"+ DATE+"_"+ PLACE +".txt", "w"):
        pass
    f = open("../prediction/"+DATE+"/"+ DATE+"_"+ PLACE +".txt", "w")
    print(PLACE,file=f)
    print(PLACE)
    y_predp = df_encoded[df_encoded['場所'] == p]
    r_list = y_predp['R'].unique()
    
    print('-----------------------',file=f)
    for i in r_list:
        print("第"+str(i) + "R",file=f)
        print("第"+str(i) + "R")
        race_pd = y_predp[y_predp["R"] == i]
        first = race_pd[race_pd['pred_rank'] == 1]['艇番'].iloc[0]
        second = race_pd[race_pd['pred_rank'] == 2]['艇番'].iloc[0]
        third = race_pd[race_pd['pred_rank'] == 3]['艇番'].iloc[0]
        print(str(first)+'-'+str(second) +'-' + str(third),file=f)
        print(str(first)+'-'+str(second) +'-' + str(third))
    print('-----------------------\n',file=f)
    print('-----------------------\n')
    f.close()

津
第11R
1-4-2
-----------------------

